# 📐 Support Vector Regression (SVR)

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_05/SVR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_05/SVR.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


### Learning objectives

By the end of this notebook, you will be able to:

- Explain the epsilon-insensitive loss.
- Understand the epsilon tube and support vectors.
- Train linear and RBF SVR models.
- Interpret the roles of `C`, `epsilon`, and `gamma`.
- Evaluate models with MAE, MSE, RMSE, and \(R^2\).
- Compare SVR briefly with ordinary linear regression.
- Apply both methods to synthetic and real datasets.

### Documentation

- [```SVR```](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html)


---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

### Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente. Puede comenzar la actividad.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

from MLTools import generate_nonlinear_regression, plot_regression_dataset

### Linear Regression Review

$$
\hat{y}= mx + b
$$

Ordinary linear regression usually minimizes the sum of squared residuals.


### Support Vector Regression

SVR searches for a function

$$
f(\mathbf{x})=\mathbf{m}^T\mathbf{x}+b
$$

while allowing an error tolerance of width $\varepsilon$.

$$
f(x)-\varepsilon \leq y \leq f(x)+\varepsilon
$$


### Epsilon-Insensitive Loss

$$
L_{\varepsilon}(y,f(x))
=
\max\left(0,\ |y-f(x)|-\varepsilon\right)
$$


In [ ]:
errors = np.linspace(-4, 4, 300)
epsilon = 1.0
loss = np.maximum(0, np.abs(errors) - epsilon)

plt.figure(figsize=(8, 5))
plt.plot(errors, loss)
plt.axvline(-epsilon, linestyle="--")
plt.axvline(epsilon, linestyle="--")
plt.xlabel("Residual")
plt.ylabel("Loss")
plt.title("Epsilon-Insensitive Loss")
plt.show()


### Main Hyperparameters

- `epsilon`: width of the tolerance tube.
- `C`: penalty for errors outside the tube.
- `gamma`: locality of nonlinear kernels.
- `kernel`: linear, polynomial, RBF, or sigmoid.


### Synthetic Nonlinear Dataset

In [ ]:
X, y, y_true = generate_nonlinear_regression(function="cubic", noise=2.2)

plot_regression_dataset( _ , _ , y_true, title="Synthetic Nonlinear Regression Dataset")


### Train Linear Regression

In [ ]:
linear_regression = LinearRegression()
linear_regression.fit( _ , _ )
linear_pred = linear_regression.predict( _ )


### Train Linear SVR

In [ ]:
linear_svr = SVR(kernel="linear", C=1.0, epsilon=1.0)

linear_svr.fit( _ , _ )
linear_svr_pred = linear_svr.predict( _ )


### Train RBF SVR

In [ ]:
rbf_svr = SVR(kernel="rbf", C=10.0, epsilon=1.0, gamma="scale")

rbf_svr.fit( _ , _ )
rbf_svr_pred = rbf_svr.predict( _ )


### Evaluation

In [ ]:
def regression_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error (y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse
    }


print( regression_metrics(y, linear_pred, "Linear Regression") ) 
print( regression_metrics(y, linear_svr_pred, "Linear SVR") ) 
print( regression_metrics(y, rbf_svr_pred, "RBF SVR") ) 

### Compare Fitted Functions

In [ ]:
x_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 500).reshape(-1, 1)

plt.figure(figsize=(10, 7))
plt.scatter(X[:, 0], y, alpha=0.5, label="Training data")
plt.plot(x_grid[:, 0], linear_regression.predict(x_grid), linewidth=2, label="Linear Regression")
plt.plot(x_grid[:, 0], linear_svr.predict(x_grid), linewidth=2, label="Linear SVR")
plt.plot(x_grid[:, 0], rbf_svr.predict(x_grid), linewidth=2, label="RBF SVR")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression vs SVR")
plt.legend()
plt.show()


### Epsilon Tube

In [ ]:
rbf_curve = rbf_svr.predict(x_grid)
epsilon_value = rbf_svr.epsilon

plt.figure(figsize=(10, 7))
plt.scatter(X[:, 0], y, alpha=0.5, label="Training data")
plt.plot(x_grid[:, 0], rbf_curve, linewidth=2, label="RBF SVR")
plt.plot(x_grid[:, 0], rbf_curve + epsilon_value, linestyle="--", label="+ epsilon")
plt.plot(x_grid[:, 0], rbf_curve - epsilon_value, linestyle="--", label="- epsilon")
plt.xlabel("x")
plt.ylabel("y")
plt.title("RBF SVR Epsilon Tube")
plt.legend()
plt.show()


### Support Vectors

In [ ]:
support_vectors = rbf_svr.support_vectors_
support_targets = y[rbf_svr.support_]

plt.figure(figsize=(10, 7))
plt.scatter(X[:, 0], y, alpha=0.4, label="Training data")
plt.scatter(
    support_vectors[:, 0],
    support_targets,
    s=100,
    facecolors="none",
    edgecolors="black",
    linewidths=1.5,
    label="Support vectors"
)
plt.plot(x_grid[:, 0], rbf_curve, linewidth=2, label="RBF SVR")
plt.plot(x_grid[:, 0], rbf_curve + epsilon_value, linestyle="--")
plt.plot(x_grid[:, 0], rbf_curve - epsilon_value, linestyle="--")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Support Vectors in RBF SVR")
plt.legend()
plt.show()

print("Number of support vectors:", len(support_vectors))


### Effect of Epsilon

In [ ]:
rows = []

for epsilon in [0.1, 0.5, 1.0, 2.0, 4.0]:
    estimator = SVR(kernel="rbf", C=10.0, epsilon=epsilon)
    estimator.fit( _ , _ )
    pred = estimator.predict( _ )

    rows.append({
        "epsilon": epsilon,
        "MSE": mean_squared_error (y, pred),
        "MAE": mean_absolute_error(y, pred),
        "Support vectors": len(estimator.support_)
    })

epsilon_df = pd.DataFrame(rows)
epsilon_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epsilon_df["epsilon"], epsilon_df["Support vectors"], marker="o")
plt.xlabel("epsilon")
plt.ylabel("Support vectors")
plt.title("Effect of Epsilon")
plt.show()


### Effect of Gamma

In [ ]:
gamma_models = {}

for gamma in [0.01, 0.1, 1, 10]:
    estimator = SVR(kernel="rbf", C=10.0, epsilon=1.0, gamma=gamma)
    estimator.fit( _ , _ )
    gamma_models[gamma] = estimator

plt.figure(figsize=(10, 7))
plt.scatter(X[:, 0], y, alpha=0.4, label="Training data")

for gamma, current in gamma_models.items():
    plt.plot(
        x_grid[:, 0],
        current.predict(x_grid),
        linewidth=2,
        label=f"gamma={gamma}"
    )

plt.xlabel("x")
plt.ylabel("y")
plt.title("Effect of Gamma on RBF SVR")
plt.legend()
plt.show()


### Brief Comparison

| Characteristic | Linear Regression | SVR |
|---|---|---|
| Objective | Minimize squared errors | Fit an epsilon tube |
| Observations used | All | Mainly support vectors |
| Nonlinearity | Requires feature engineering | Available through kernels |
| Scaling | Often optional | Usually essential |
| Main parameters | Few | `C`, `epsilon`, `gamma`, kernel |
| Interpretability | High | Lower |
| Training cost | Usually low | Can be high |


### Practice Exercises

#### Exercise 1
Compare several values of `epsilon`.


In [ ]:
# Your code here

#### Exercise 2
Compare several values of `C`.


In [ ]:
# Your code here

#### Exercise 3
Generate a nearly linear dataset with outliers and compare Linear Regression with Linear SVR.


In [ ]:
# Your code here

#### Exercise 4
Compare linear, polynomial, RBF, and sigmoid SVR kernels.


In [ ]:
# Your code here

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Support Vector Regression (SVR): Machine Learning Notebook*. Institución Universitaria Pascual Bravo.